# 1. GENERAR DATASET

In [ ]:
import pandas as pd
import glob
import os

ruta = "PV_MaximumPowerPredictor/*.csv"

dfs = []

for archivo in glob.glob(ruta):
    
    nombre = os.path.basename(archivo).replace(".csv","")
    
    partes = nombre.split("_")
    
    parque_id = partes[0]              # Cocoa / Golden / Eugene
    panel_id = "_".join(partes[1:])    # tipo de panel
    
    df = pd.read_csv(archivo)
    
    # eliminar timestamp
    df = df.drop(columns=["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"], errors="ignore")
    
    # añadir columnas
    df["parque_id"] = parque_id
    df["panel_id"] = panel_id
    
    # mover columnas al inicio
    cols = ["parque_id","panel_id"] + [c for c in df.columns if c not in ["parque_id","panel_id"]]
    df = df[cols]
    
    dfs.append(df)

dataset_total = pd.concat(dfs, ignore_index=True)

print(dataset_total.shape)
dataset_total.head()

(1025599, 12)


,parque_id,panel_id,POA irradiance CMP22 pyranometer (W/m2),PV module back surface temperature (degC),Pmp (W),Dry bulb temperature (degC),Relative humidity (%RH),Atmospheric pressure (mb),Precipitation (mm) accumulated daily total,Direct normal irradiance (W/m2),Global horizontal irradiance (W/m2),Diffuse horizontal irradiance (W/m2)
0,Cocoa,aSiMicro03036_cleaned,20.2,19.3,1.9610,17.7,96.0,1009.8,20.3,0.1,21.9,22.2
1,Cocoa,aSiMicro03036_cleaned,35.8,19.5,3.7242,17.8,96.0,1009.7,20.3,0.1,38.4,39.0
2,Cocoa,aSiMicro03036_cleaned,20.2,19.5,1.9551,17.8,96.0,1009.9,20.3,0.1,20.1,20.3
3,Cocoa,aSiMicro03036_cleaned,20.6,19.5,2.0057,17.8,96.1,1009.8,20.3,0.1,21.4,21.8
4,Cocoa,aSiMicro03036_cleaned,29.5,19.5,3.1397,17.8,96.0,1009.9,20.6,0.1,29.3,29.5


# 2. EXPLORAR LOS DATOS

In [ ]:
dataset_total["parque_id"].value_counts() # Comprobar num de parques y paneles

parque_id
Eugene    473348
Cocoa     421664
Golden    130587
Name: count, dtype: int64

In [ ]:
dataset_total["panel_id"].nunique()

22

In [ ]:
sorted(dataset_total["panel_id"].unique())

['CIGS1-001_cleaned',
 'CIGS39013_cleaned',
 'CIGS39017_cleaned',
 'CIGS8-001_cleaned',
 'CdTe75638_cleaned',
 'CdTe75669_cleaned',
 'HIT05662_cleaned',
 'HIT05667_cleaned',
 'aSiMicro03036_cleaned',
 'aSiMicro03038_cleaned',
 'aSiTandem72-46_cleaned',
 'aSiTandem90-31_cleaned',
 'aSiTriple28324_cleaned',
 'aSiTriple28325_cleaned',
 'mSi0166_cleaned',
 'mSi0188_cleaned',
 'mSi0247_cleaned',
 'mSi0251_cleaned',
 'mSi460A8_cleaned',
 'mSi460BB_cleaned',
 'xSi11246_cleaned',
 'xSi12922_cleaned']

In [ ]:
pd.crosstab(dataset_total["parque_id"], dataset_total["panel_id"])

panel_id,CIGS1-001_cleaned,CIGS39013_cleaned,CIGS39017_cleaned,CIGS8-001_cleaned,CdTe75638_cleaned,CdTe75669_cleaned,HIT05662_cleaned,HIT05667_cleaned,aSiMicro03036_cleaned,aSiMicro03038_cleaned,...,aSiTriple28324_cleaned,aSiTriple28325_cleaned,mSi0166_cleaned,mSi0188_cleaned,mSi0247_cleaned,mSi0251_cleaned,mSi460A8_cleaned,mSi460BB_cleaned,xSi11246_cleaned,xSi12922_cleaned
parque_id,,,,,,,,,,,,,,,,,,,,,
Cocoa,0,0,34775,38939,39080,0,0,38377,39037,0,...,38485,0,36765,39102,0,0,38929,0,0,38989
Eugene,0,0,42674,43146,42248,0,0,43271,43343,0,...,42705,0,43268,43127,0,0,43115,0,0,43185
Golden,12011,11437,0,0,0,11953,11876,0,0,12148,...,0,11445,0,0,11912,11887,0,11919,11929,0


In [ ]:
dataset_total.isna().sum()

parque_id                                     0
panel_id                                      0
POA irradiance CMP22 pyranometer (W/m2)       0
PV module back surface temperature (degC)     0
Pmp (W)                                       0
Dry bulb temperature (degC)                   0
Relative humidity (%RH)                       0
Atmospheric pressure (mb)                     0
Precipitation (mm) accumulated daily total    0
Direct normal irradiance (W/m2)               0
Global horizontal irradiance (W/m2)           0
Diffuse horizontal irradiance (W/m2)          0
dtype: int64

# 3. MODELO BASELINE

Ignoramos el parque y el panel y también el timestamp.

In [ ]:
df_model = dataset_total.drop(columns=["parque_id", "panel_id"])

In [ ]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train.shape


(820479, 9)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import numpy as np

np.mean(X_train_scaled, axis=0)


array([-2.50709795e-17,  4.06751745e-16, -4.57599502e-17, -9.89329430e-17,
       -5.29651505e-17,  5.30344313e-17, -2.94443283e-17,  9.56074659e-18,
       -1.18781884e-16])

In [ ]:
np.std(X_train_scaled, axis=0)

array([1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [ ]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

## A. DISEÑO DEL MODELO

In [ ]:
import torch
import torch.nn as nn

class PVModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(9, 64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 16),
            nn.ReLU(),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [ ]:
model = PVModel()
print(model)

PVModel(
  (model): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [ ]:
model(X_train_tensor[:5])

tensor([[ 0.2366],
        [ 0.2439],
        [-0.0108],
        [ 0.2341],
        [ 0.2291]], grad_fn=<AddmmBackward0>)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

## B. ML FLOW

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("PV_Power")

c:\Users\User\Documents\GitHub\Reto3_MicroRedes\.venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///c:/Users/User/Documents/GitHub/Reto3_MicroRedes/OBJETIVO2/mlruns/427506226560148484', creation_time=1773741607353, experiment_id='427506226560148484', last_update_time=1773741607353, lifecycle_stage='active', name='PV_Power', tags={}, workspace='default'>

In [ ]:
class PVModel(nn.Module):
    
    def __init__(self, layers_sizes):
        super().__init__()
        
        layers = []
        input_size = 9
        
        for size in layers_sizes:
            layers.append(nn.Linear(input_size, size))
            layers.append(nn.ReLU())
            input_size = size
        
        layers.append(nn.Linear(input_size, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

In [ ]:
def evaluate(model, loader, criterion):
    
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [ ]:
def train_model(arch, lr, epochs):
    
    model = PVModel(arch)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    patience = 5
    counter = 0
    
    with mlflow.start_run():
    
        mlflow.log_param("arquitectura", str(arch))
        mlflow.log_param("lr", lr)
        
        for epoch in range(epochs):
            
            model.train()
            total_loss = 0
            
            for X_batch, y_batch in train_loader:
                
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            train_loss = total_loss / len(train_loader)
            val_loss = evaluate(model, test_loader, criterion)
            
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            
            print(f"Epoch {epoch+1} | Train: {train_loss:.2f} | Val: {val_loss:.2f}")

    return model

In [ ]:
arquitecturas = [
    [64, 32, 16],
    [128, 64, 32],
    [256, 128, 64]
]

learning_rates = [0.001, 0.0005]

epochs = 100

In [ ]:
# Aqui entrenaba pero no había mejora

# 4. FEATURE ENGENIERING

In [ ]:
df_model = dataset_total.drop(columns=["parque_id", "panel_id"])

In [ ]:
corr = df_model.corr(numeric_only=True)
corr["Pmp (W)"].sort_values(ascending=False)

In [ ]:
df_model["physical_model"] = (
    df_model["POA irradiance CMP22 pyranometer (W/m2)"] *
    (1 - 0.004 * (df_model["PV module back surface temperature (degC)"] - 25))
)

In [ ]:
df_model = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "physical_model",
    "Pmp (W)"
]]

In [ ]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

In [51]:
for arch in arquitecturas:
    for lr in learning_rates:
        
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        
        train_model(arch, lr, epochs)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 837.05 | Val: 623.23
Epoch 2 | Train: 618.87 | Val: 621.59
Epoch 3 | Train: 618.12 | Val: 622.77
Epoch 4 | Train: 618.18 | Val: 621.58
Epoch 5 | Train: 617.99 | Val: 619.78
Epoch 6 | Train: 617.92 | Val: 621.30
Epoch 7 | Train: 617.93 | Val: 620.02
Epoch 8 | Train: 617.71 | Val: 620.23
Epoch 9 | Train: 617.56 | Val: 619.72
Epoch 10 | Train: 617.66 | Val: 619.98
Epoch 11 | Train: 617.81 | Val: 621.81
Epoch 12 | Train: 617.62 | Val: 620.05
Epoch 13 | Train: 617.49 | Val: 620.04
Epoch 14 | Train: 617.57 | Val: 619.73
Epoch 15 | Train: 617.62 | Val: 619.64
Epoch 16 | Train: 617.80 | Val: 619.60
Epoch 17 | Train: 617.69 | Val: 619.79
Epoch 18 | Train: 617.56 | Val: 621.11
Epoch 19 | Train: 617.36 | Val: 619.69
Epoch 20 | Train: 617.56 | Val: 620.94
Epoch 21 | Train: 617.61 | Val: 620.77
Epoch 22 | Train: 617.59 | Val: 619.75
Epoch 23 | Train: 617.24 | Val: 620.62
Epoch 24 | Train: 617.49 | Val: 619.53
Epoch 25 | Train: 617.49 | V

KeyboardInterrupt: 

In [52]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Entrenar modelo
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predicción
y_pred = lr_model.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 621.2737923638234
RMSE: 24.925364437933972
R2: 0.5601160592375038


In [53]:
y_pred_nn = model(X_test_tensor).detach().numpy()

In [54]:
mean_squared_error(y_test, y_pred_nn)

2509.8810446553075

# 5. MODELO SIMPLE:

Los modelos complejos no funcionan, vamos a probar los simples.

In [55]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Modelo
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entrenar
model.fit(X_train, y_train)

# Predecir
y_pred = model.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 621.7360079178163
RMSE: 24.93463470592293
R2: 0.5597887941864568


# 6. DEPENDENDIA TEMPORAL

In [57]:
ruta = "PV_MaximumPowerPredictor/*.csv"

dfs = []

for archivo in glob.glob(ruta):
    
    nombre = os.path.basename(archivo).replace(".csv","")
    
    partes = nombre.split("_")
    
    parque_id = partes[0]              # Cocoa / Golden / Eugene
    panel_id = "_".join(partes[1:])    # tipo de panel
    
    df = pd.read_csv(archivo)

    
    # añadir columnas
    df["parque_id"] = parque_id
    df["panel_id"] = panel_id
    
    # mover columnas al inicio
    cols = ["parque_id","panel_id"] + [c for c in df.columns if c not in ["parque_id","panel_id"]]
    df = df[cols]
    
    dfs.append(df)

dataset_total = pd.concat(dfs, ignore_index=True)

print(dataset_total.shape)
dataset_total.head()

(1025599, 13)


,parque_id,panel_id,Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss,POA irradiance CMP22 pyranometer (W/m2),PV module back surface temperature (degC),Pmp (W),Dry bulb temperature (degC),Relative humidity (%RH),Atmospheric pressure (mb),Precipitation (mm) accumulated daily total,Direct normal irradiance (W/m2),Global horizontal irradiance (W/m2),Diffuse horizontal irradiance (W/m2)
0,Cocoa,aSiMicro03036_cleaned,2011-01-21T08:10:26,20.2,19.3,1.9610,17.7,96.0,1009.8,20.3,0.1,21.9,22.2
1,Cocoa,aSiMicro03036_cleaned,2011-01-21T08:15:26,35.8,19.5,3.7242,17.8,96.0,1009.7,20.3,0.1,38.4,39.0
2,Cocoa,aSiMicro03036_cleaned,2011-01-21T08:20:26,20.2,19.5,1.9551,17.8,96.0,1009.9,20.3,0.1,20.1,20.3
3,Cocoa,aSiMicro03036_cleaned,2011-01-21T08:25:26,20.6,19.5,2.0057,17.8,96.1,1009.8,20.3,0.1,21.4,21.8
4,Cocoa,aSiMicro03036_cleaned,2011-01-21T08:35:26,29.5,19.5,3.1397,17.8,96.0,1009.9,20.6,0.1,29.3,29.5


In [58]:
df_model = dataset_total.copy()

In [59]:
df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"] = pd.to_datetime(
    df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"]
)

In [60]:
df_model["hour"] = df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"].dt.hour
df_model["day_of_year"] = df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"].dt.dayofyear

In [61]:
import numpy as np

df_model["hour_sin"] = np.sin(2 * np.pi * df_model["hour"] / 24)
df_model["hour_cos"] = np.cos(2 * np.pi * df_model["hour"] / 24)

df_model["day_sin"] = np.sin(2 * np.pi * df_model["day_of_year"] / 365)
df_model["day_cos"] = np.cos(2 * np.pi * df_model["day_of_year"] / 365)

In [62]:
df_model = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "Pmp (W)"
]]

In [63]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [65]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [66]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = mse_lr ** 0.5
r2_lr = r2_score(y_test, y_pred_lr)

print("LINEAR REGRESSION")
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)
print("R2:", r2_lr)

LINEAR REGRESSION
MSE: 621.3078811180261
RMSE: 24.926048245119524
R2: 0.5600919231871526


In [67]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [68]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024)

In [69]:
import torch.nn as nn

class PVModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [70]:
def evaluate(model, loader, criterion):
    
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [71]:
model = PVModel()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50
best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    train_loss = total_loss / len(train_loader)
    val_loss = evaluate(model, test_loader, criterion)
    
    print(f"Epoch {epoch+1} | Train: {train_loss:.2f} | Val: {val_loss:.2f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
    else:
        counter += 1
    
    if counter >= patience:
        print(" Early stopping")
        break

Epoch 1 | Train: 1017.45 | Val: 628.63
Epoch 2 | Train: 620.54 | Val: 621.05
Epoch 3 | Train: 618.49 | Val: 620.41
Epoch 4 | Train: 618.09 | Val: 620.23
Epoch 5 | Train: 618.00 | Val: 620.47
Epoch 6 | Train: 617.84 | Val: 620.00
Epoch 7 | Train: 618.01 | Val: 620.42
Epoch 8 | Train: 617.70 | Val: 620.05
Epoch 9 | Train: 617.87 | Val: 619.85
Epoch 10 | Train: 617.60 | Val: 619.86
Epoch 11 | Train: 617.79 | Val: 620.11
Epoch 12 | Train: 617.87 | Val: 619.84
Epoch 13 | Train: 617.65 | Val: 620.08
Epoch 14 | Train: 617.62 | Val: 619.98
Epoch 15 | Train: 617.82 | Val: 619.95
Epoch 16 | Train: 617.53 | Val: 620.54
Epoch 17 | Train: 617.74 | Val: 620.10
⛔ Early stopping


In [72]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred_nn = model(X_test_tensor).detach().numpy()

mse_nn = mean_squared_error(y_test, y_pred_nn)
rmse_nn = mse_nn ** 0.5
r2_nn = r2_score(y_test, y_pred_nn)

print("\nNEURAL NETWORK")
print("MSE:", mse_nn)
print("RMSE:", rmse_nn)
print("R2:", r2_nn)


NEURAL NETWORK
MSE: 620.7592895559866
RMSE: 24.915041431954045
R2: 0.5604803455238825


# 7. PRUEBAS VARIAS A VER SI ALGO MEJORA EL LOSS

In [73]:
# ================================
# 1. PREPARAR DATASET
# ================================
import pandas as pd
import numpy as np

df_model = dataset_total.copy()

# Convertir timestamp
df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"] = pd.to_datetime(
    df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"]
)

# Features temporales
df_model["hour"] = df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"].dt.hour
df_model["day_of_year"] = df_model["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"].dt.dayofyear

df_model["hour_sin"] = np.sin(2 * np.pi * df_model["hour"] / 24)
df_model["hour_cos"] = np.cos(2 * np.pi * df_model["hour"] / 24)

df_model["day_sin"] = np.sin(2 * np.pi * df_model["day_of_year"] / 365)
df_model["day_cos"] = np.cos(2 * np.pi * df_model["day_of_year"] / 365)

# ================================
# 2. MODELO FÍSICO
# ================================
k = 0.004  # coeficiente típico

df_model["physical_model"] = (
    df_model["POA irradiance CMP22 pyranometer (W/m2)"] *
    (1 - k * (df_model["PV module back surface temperature (degC)"] - 25))
)

# ================================
# 3. TARGET = ERROR (RESIDUAL)
# ================================
df_model["residual"] = df_model["Pmp (W)"] - df_model["physical_model"]

# ================================
# 4. SELECCIÓN DE VARIABLES
# ================================
X = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]]

y = df_model["residual"]

# ================================
# 5. SPLIT
# ================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, phys_train, phys_test = train_test_split(
    X, y, df_model["physical_model"], test_size=0.2, random_state=42
)

# ================================
# 6. ESCALADO
# ================================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ================================
# 7. MODELO (XGBOOST)
# ================================
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# ================================
# 8. PREDICCIÓN FINAL
# ================================
residual_pred = model.predict(X_test_scaled)

# Reconstrucción final
y_pred_final = phys_test + residual_pred

# ================================
# 9. MÉTRICAS
# ================================
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(df_model.loc[y_test.index, "Pmp (W)"], y_pred_final)
rmse = mse ** 0.5
r2 = r2_score(df_model.loc[y_test.index, "Pmp (W)"], y_pred_final)

print("\nMODELO HÍBRIDO (FÍSICA + ML)")
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)


MODELO HÍBRIDO (FÍSICA + ML)
MSE: 637.8880387510865
RMSE: 25.256445489242672
R2: 0.5483525819051966


In [74]:
# ================================
# POLYNOMIAL FEATURES + REGRESSION
# ================================
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Variables base (las buenas)
X = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos"
]]

y = df_model["Pmp (W)"]

# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ================================
# POLYNOMIAL FEATURES
# ================================
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# ================================
# MODELO
# ================================
model = LinearRegression()
model.fit(X_train_poly, y_train)

# Predicción
y_pred = model.predict(X_test_poly)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("\nPOLYNOMIAL REGRESSION")
print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)


POLYNOMIAL REGRESSION
MSE: 620.5781359350717
RMSE: 24.911405739842778
R2: 0.5606086087302673
